# Grid Up Datathon — 02 · Baseline

Amaç: **en hızlı geçerli submission** — ama rastgele değil, ölçülmüş bir planla.
Yarışmadan önce, 68.257 gerçek GDZ kesinti kaydında (47 ilçe, 2021-05 → 2022-08)
hangi feature ailesinin katkı verdiğini ve hangi model reçetesinin kazandığını
**aynı purged fold'larda** ölçtük (`scripts/ablation_gercek.py`,
`scripts/benchmark_gercek.py`). Veri gününde deney değil **icra** yapacağız:
plan sabittir, sayılar yarışma verisinde yeniden ölçülür.

Sıra: fold'lar → feature'lar (ölçülen öncelikle) → model (ölçülen reçete) →
harman → submission.

In [ ]:
import sys
from pathlib import Path

# Kaggle'da: /kaggle/input/<yarisma>/  · yerelde: data/raw/
IS_KAGGLE = Path("/kaggle/input").exists()
if IS_KAGGLE:
    # DIKKAT: gridup Kaggle imajinda KURULU DEGILDIR. Onceki surumde sys.path
    # yalnizca YERELDE ayarlaniyordu; Kaggle'da 'import gridup' ModuleNotFound
    # veriyordu. Once offline paket dataset'indeki wheel'i kur, o yoksa ham
    # kaynagi sys.path'e ekle. (Path.glob, glob.glob degil: juri notebook'u
    # ruff'tan geciyor -- PTH207.)
    import subprocess

    _whl = sorted(Path("/kaggle/input").glob("*/gridup-*.whl"))
    if _whl:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
             str(_whl[0]), "-q"],
            check=False,
        )
    else:
        for _src in Path("/kaggle/input").glob("*/src"):
            sys.path.insert(0, str(_src))
else:
    sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from gridup import (
    conditional_quantile_from_hurdle, cross_validate, fit_conditional_quantile_ladder,
    fit_two_stage, make_model_zoo, read_any, set_global_seed,
    sweep_count_objectives, tune_with_optuna, write_submission, zero_baseline_score,
)
from gridup.compat import categorical_columns
from gridup.ensemble import hill_climb_weights, prune_by_correlation, stack_oof
from gridup.experiment import ExperimentLog, ExperimentRecord
from gridup.features import (
    add_calendar_features, add_frequency_encoding, add_lag_features,
    add_neighbour_target_lag, add_physical_derivatives, add_regional_aggregates,
    nearest_neighbours, shared_origin,
)
from gridup.metrics import inverse_log_transform, log_transform_target
from gridup.models import starter_params
from gridup.refit import (estimate_full_data_rounds, extract_best_iterations,
                          fold_train_fraction, multi_seed_refit)
from gridup.selection import null_importance_filter, shap_backward_selection
from gridup.validation import adversarial_validation, build_splitter, purged_time_series_split

set_global_seed(42)

DATA_DIR = Path("/kaggle/input/GRID-UP-YARISMA-SLUG") if IS_KAGGLE else Path("../data/raw")
OUT_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("../submissions")

TARGET = "HEDEF_KOLON"     # TODO
ID_COLUMN = "id"           # TODO
TIME_COLUMN = None         # TODO
GROUP_COLUMN = None        # TODO
METRIC = "rmse"            # TODO — yarışmanın resmi metriği (2024'te MAE idi!)
TASK = "regression"        # regression | binary | multiclass
LOG_TARGET = False         # metrik RMSLE ise veya hedef çok çarpıksa True

# TAHMİN UFKU — en pahalı sessiz hatanın kaynağı.
# Test ilerideki bir BLOK ise (ör. bir sonraki ay), o bloğun son gününü
# tahmin ederken elindeki en taze veri blok uzunluğu kadar eskidir.
# shift(1) ile hesaplanan lag'ler CV'de harika görünür, private LB'de çöker.
# Veri geldiğinde: HORIZON = (test.tarih.max() - test.tarih.min()).days + 1
HORIZON = 1                # TODO

In [ ]:
train = read_any(DATA_DIR / "train.csv")
test  = read_any(DATA_DIR / "test.csv")
print(train.shape, test.shape)

## 1 · Fold'lar — feature üretmeden önce

Hedef kodlama ve lag'ler fold'lara ihtiyaç duyar; fold'lar önce sabitlenir ki
bugünün ve yarının **bütün** deneyleri aynı bölmeler üzerinde karşılaştırılabilsin.
Şemanın gerekçesi 01 numaralı notebook'ta; özeti: `test_span` = tahmin ufku
(2023 birincisinin `test_size=744` paraleli), `embargo` bilinçli ve ufuktan küçük
değil.

In [ ]:
if TIME_COLUMN:
    train[TIME_COLUMN] = pd.to_datetime(train[TIME_COLUMN])
    test[TIME_COLUMN] = pd.to_datetime(test[TIME_COLUMN])
    HORIZON = int((test[TIME_COLUMN].max() - test[TIME_COLUMN].min()).days) + 1
    print(f"Tahmin ufku (test blok uzunlugu): {HORIZON} gun")
    # test_span = ufuk: fold'lar zaman uzunlugu esit pencereler olsun.
    # embargo >= ufuk: kayan pencereler fold sinirini asmasin.
    folds = purged_time_series_split(
        train[TIME_COLUMN], n_splits=4,
        embargo=pd.Timedelta(days=max(HORIZON, 30)),
        test_span=pd.Timedelta(days=HORIZON),
    )
elif GROUP_COLUMN:
    splitter = build_splitter("GroupKFold", n_splits=5)
    folds = list(splitter.split(train, groups=train[GROUP_COLUMN]))
else:
    scheme = "StratifiedKFold" if TASK != "regression" else "KFold"
    splitter = build_splitter(scheme, n_splits=5, seed=42)
    folds = list(splitter.split(train, train[TARGET] if TASK != "regression" else None))

for i, (tr, va) in enumerate(folds, 1):
    print(f"fold {i}: train={len(tr):>8,}  valid={len(va):>8,}")

## 2 · Feature aileleri — önceliği tahmin değil ölçüm belirledi

Feature önemi (gain) aile önceliğini **söyleyemez**: korele kolonlar tek tek
"önemli" görünür ama biri silinince diğeri işi devralır. Bu yüzden ölçü
leave-one-group-out'tur: aile tümüyle silinir, aynı purged fold'larla MAE yeniden
ölçülür. Gerçek GDZ verisinde, 76 feature'lı tam model MAE 313.64 / hep-sıfır
366.97 iken (`scripts/ablation_gercek.py` → `experiments/ablasyon_gercek.json`):

| Aile | Δ MAE (silinince kayıp) | Kolon | Veri günü kararı |
|---|---|---|---|
| lag / rolling | **+22.34** | 11 | İlk kurulacak — kalanların toplamından büyük |
| hava | +2.47 | 24 | İkinci; ortalama değil `max`/quantile agregatları |
| komşu lag | +0.18 | 3 | Marjinal; ucuz, ufuk şartıyla kalır |
| frekans | 0.00 | 1 | Tam ızgara panelde sabit 1/47 → sıfır bilgi |
| takvim | −0.19 | 15 | Gürültü bandında |
| tatil | −4.66 | 15 | Silinince MAE düşüyor → ilk elenecek aday |
| güneş | −5.03 | 7 | İlk elenecek aday |

Tam modelin en yüksek gain'li kolonu da aynı hikâyeyi anlatıyor: 93 günlük,
ufuk-kaydırmalı kayan ortalama (`kesinti_dk_ufuk31_kayan93_mean`). Geçmiş kesinti
davranışı en güçlü sinyaldir — ama ancak ufuk kadar kaydırılmışsa meşrudur.

Ablasyonun fold_std'si 94.31, skorun ~%30'u: küçük deltalar kesin hüküm değildir.
Sıralamanın ucu ise (lag ≫ diğerleri) gürültünün çok üstünde.

In [ ]:
# ORTAK zaman baslangici: train ve test icin ayri ayri hesaplanirsa test'in
# gun sayaci yeniden 0'dan baslar ve model test'i train'in gecmisi sanir.
# Bu hata lokal CV'de GORUNMEZ -- sadece leaderboard coker.
ORIGIN = shared_origin(train, test, time_column=TIME_COLUMN) if TIME_COLUMN else None

def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Train ve test'e ayni donusumleri uygular. Girdiyi degistirmez."""
    out = frame.copy()
    if TIME_COLUMN:
        out = add_calendar_features(out, TIME_COLUMN, include_year=False, origin=ORIGIN)
    # categorical_columns: pandas 2.x ve 3.x'te de dogru calisir.
    # Duz `dtype == object` kontrolu pandas 3.0'da metin kolonlarini KACIRIR.
    categorical = categorical_columns(out)
    if categorical:
        out = add_frequency_encoding(out, categorical[:12])
    return out

train_features = build_features(train)
test_features = build_features(test)

# LAG AILESI -- ablasyonda tek basina en buyuk katki (+22.34 MAE).
# Train + test BIRLIKTE kurulur: test satirlarinin lag'i train'in son
# gunlerinden gelir. horizon=HORIZON kaydirma sayesinde hicbir satir kendi
# gununun (veya daha yakinin) bilgisini goremez -- sizinti duvari korunur.
if TIME_COLUMN and GROUP_COLUMN:
    n_train = len(train_features)
    butun = pd.concat([train_features, test_features], ignore_index=True, sort=False)
    butun = add_lag_features(
        butun, TARGET, [HORIZON, 2 * HORIZON, 3 * HORIZON],
        time_column=TIME_COLUMN, group_columns=[GROUP_COLUMN], horizon=HORIZON,
    )
    train_features = butun.iloc[:n_train].reset_index(drop=True)
    test_features = butun.iloc[n_train:].reset_index(drop=True)

drop = {TARGET, ID_COLUMN, TIME_COLUMN} - {None}
FEATURES = [c for c in train_features.columns
            if c not in drop and c in test_features.columns]
print(f"{len(FEATURES)} feature")

## 3 · Model reçetesi — dokuz aday, aynı fold'lar, aynı bütçe

2023 GDZ birincisi CatBoost'u MAE kaybıyla, 2024 birincisi (Pikachow) LightGBM'i
Optuna ile kullandı; ama o seçimler o yılların verisinde yapıldı. Dokuz reçeteyi
gerçek GDZ verisinde aynı fold, aynı feature seti ve aynı ağaç bütçesiyle
yarıştırdık (`scripts/benchmark_gercek.py` → `experiments/benchmark_gercek.json`):

| Reçete | MAE (dk) | Not |
|---|---|---|
| **catboost_mae** | **304.89** | Kazanan tekil: 2023 birincisinin reçetesi zirveye çıktı |
| lgb_mae | 310.14 | En iyi LightGBM — kayıp = metrik |
| iki_asama_medyan | 316.95 | Koşullu merdiven + q\*=1−0.5/p kuralı |
| iki_asama (eşikli) | 317.04 | Eşik 0.680; medyan kuralının farkı 0.1 dk'ya indi |
| lgb_sqrt | 324.03 | Rohlik reçetesi: sqrt(y)+L2, geri-kare |
| lgb_tweedie | 327.83 | Sıfır-şişkin hedefe uygun, hızlı aday — harmanda değerli |
| iki_asama_medyan_kalibre | 328.15 | İzotonik kalibrasyon BOZDU (Brier 0.207→0.241) |
| xgb | 402.83 | Baseline'ın altında |
| lgb_l2 | 403.78 | Metrik MAE iken L2 kaybı ~94 dk kaybettiriyor |
| hep-sıfır | 366.97 | Alt çizgi; bunu geçemeyen model rafa kalkar |

Bu tablo **3. dalga** feature setiyle ölçüldü: önceki 49 kolona Hawkes-esinli
üstel bozunum (3g/14g yarı ömür) ve bölgesel toplu-olay payı eklendi (56 kolon,
ikisi de ufuk=31 kaydırmalı) — sayıların önceki dalgadan (en iyi tekil 312.74)
oynamasının nedeni bu. Örnek ağırlığı (`recency_activity_weights`) tek başına
kazandırmıştı ama Hawkes ile ÇATIŞTI (aynı yenilik sinyali iki kanaldan →
lgb_mae 310.14→335.30); ölçüm sonucu kanonik koşu ağırlıksız.

Dört ölçülmüş ders:

- **Kayıp fonksiyonu model seçiminden önce gelir.** Aynı LightGBM, kayıp
  `l2 → mae` değişince ~94 dk kazanıyor. Yarışma metriği neyse kayıp odur.
- **MAE'nin optimali medyandır ama kazancı feature setine bağlıdır.** Karışımın
  medyanı: p≤0.5 ise tam 0, değilse koşullu dağılımın q\*=1−0.5/p kantili
  (`conditional_quantile_from_hurdle`). Önceki dalgada eşikli hurdle'ı 4.5 dk
  geçmişti (317.23→312.74); Hawkes feature'ları eklenince fark 0.1 dk'ya indi —
  kural bedava, ama sinyali feature'lar zaten taşıyorsa kazanç erir.
- **Kalibrasyonu varsaymadık, ölçtük.** "Eşik 0.680, sınıflandırıcı kalibresiz
  olduğu için 0.5'ten sapmış olabilir" hipotezini izotonik kalibrasyonla test
  ettik: Brier kötüleşti, MAE kötüleşti. Eşik sapması verinin gerçeği.
- **Bu tablonun ilk sürümü sızıntılıydı ve bunu çekişmeli denetim yakaladı.**
  Ham kaydın `id` kolonu panel dolgusunun birebir kopyası çıktı (y==0 ile uyum
  0.9975) ve tüm skorları ~60 dk iyimser gösterdi. Yukarıdaki sayılar, `id`
  dahil ham olay kolonlarının tamamı sızıntı duvarının arkasına alındıktan
  sonraki dürüst ölçümdür.

In [ ]:
y = train_features[TARGET].to_numpy()
if LOG_TARGET:
    y = log_transform_target(y)

# Kayip = yarisma metrigi. Gercek GDZ olcumu: ayni LightGBM'de l2 -> mae
# gecisi ~94 dk kazandirdi (experiments/benchmark_gercek.json).
params = (starter_params("lightgbm", TASK, objective="mae") if METRIC == "mae"
          else starter_params("lightgbm", TASK))

result = cross_validate(
    train_features[FEATURES], y, folds,
    kind="lightgbm", task_type=TASK, metric=METRIC,
    params=params, test=test_features[FEATURES],
)

print(result.summary())

## 4 · Harman — ve kapsam maskesi neden şart

Gerçek GDZ ölçümünde en iyi tekil model 304.89; TÜM üyeler üzerinde
kararlılık-cezalı hill-climb harmanı **302.64** (ağırlık alanlar: catboost_mae
0.75 + lgb_tweedie 0.25; `stability_penalty=0.5` — tırmanma tek fold'un
hediyesini değil, fold MAE'lerinin ortalama + 0.5·std'sini kovalar) —
`scripts/benchmark_gercek.py`. Bir ölçülmüş ders daha: "en iyi 3 üyeyi
harmanla" kısayolu bir önceki dalgada 311.83 verdi, çünkü en iyi üçü birbirinin
kopyası çıktı — harmanı üye kalitesi değil **hata çeşitliliği** taşır; hill-climb
tüm adayları görünce işe yaramayana zaten 0 ağırlık veriyor (lgb_tweedie tekil
6. sıradayken harmanda ağırlık alan iki üyeden biri olması bunun kanıtı).
Ridge stacking ise rekabet dışı: purged şemada ilk dönem hiçbir fold'un valid
tarafına düşmediği için meta-modelin eğitim kapsamı daralıyor. Tercihimiz hill
climbing — ağırlıklar jüriye tek satırda açıklanabilir.

**Kapsam maskesi:** purged bölme ilk dönemi hiçbir valid'e koymaz; o satırların
OOF değeri tahmin değil **dolgudur** (0.0). Maskesiz harman kurmak skoru ölçülü
biçimde şişirir: rmse 2.213 → 2.755, **%24.5 sapma** (`src/gridup/ensemble.py`,
`tests/test_harman_kapsami.py`). Bu yüzden harman/stack her zaman `covered`
maskesi üzerinde kurulur.

In [ ]:
zoo = make_model_zoo(train_features[FEATURES], y, folds, metric=METRIC,
                     test=test_features[FEATURES])

# KAPSAM MASKESI SART: purged ilk donemi hicbir fold'un valid tarafina koymaz;
# o satirlarda OOF degeri dolgudur ve harman skorunu %24.5'e kadar sisirir
# (olculdu, ensemble.py). covered_oof_matrix maskeyi otomatik uygular.
kapsanan, oof = zoo.covered_oof_matrix()
y_kapsanan = y[kapsanan]
print(f"OOF kapsami: %{zoo.coverage * 100:.1f}")

secilen = prune_by_correlation(oof, y_kapsanan, max_members=5)
agirliklar = hill_climb_weights({k: oof[k] for k in secilen}, y_kapsanan, metric=METRIC)
print("harman agirliklari:", agirliklar)

# Stacking'i ancak fold kapsami genisse dene -- gercek GDZ'de purged semada
# 645.48 ile rekabet disiydi (benchmark_gercek.json):
# stack = stack_oof(zoo.oof_matrix, y, folds, test_predictions=zoo.test_matrix,
#                   base_covered=zoo.oof_covered)

## 5 · Submission

`write_submission` yazmadan önce doğrular: NaN, sonsuz, eksik ID, sabit tahmin,
negatif değer. Kaggle'ın "Submission Scoring Error" mesajı hiçbir şey söylemez;
hatayı gönderMEDEN yakalamak bir submission hakkı kurtarır.

In [ ]:
predictions = result.test_predictions
if LOG_TARGET:
    predictions = inverse_log_transform(predictions)

path = write_submission(
    test_features[ID_COLUMN].to_numpy(),
    predictions,
    OUT_DIR / "baseline_lgbm.csv",
    id_column=ID_COLUMN,
    target_column=TARGET,
)

## 6 · Deney defteri

Submission gönderdikten **sonra** leaderboard skorunu geri yaz:

```python
log.record_lb("baseline_lgbm", 12.3456)
print(log.cv_lb_correlation())
```

CV–LB korelasyonu bu yarışmanın en önemli tek sayısıdır. r > 0.8 ise CV'ne
güven; r < 0.5 ise CV şemanı düzeltmeden devam etme.

In [ ]:
log = ExperimentLog(OUT_DIR.parent / "experiments" / "deneyler.jsonl")

log.add(ExperimentRecord(
    name="baseline_lgbm",
    cv_score=result.overall_score,
    metric=METRIC,
    model_kind="lightgbm",
    n_features=len(FEATURES),
    fold_scores=result.fold_scores,
    notes="baseline: takvim + frekans + ufuk-kaydirmali lag",
    submission_path=str(path),
))

log.leaderboard()

## 7 · Dürüst sınırlar — bu sayıların söylemediği şeyler

- **CV gürültülü.** Provada tek modelin fold skorları 150.8 → 461.4 dk salındı
  (std 122.35, `scripts/real_data_rehearsal.py`); ablasyonda fold_std 94.31,
  benchmark'ta 22.4–113.1. İki reçete arasındaki 2–3 dakikalık fark hüküm
  değildir; kararlar aile düzeyindeki büyük farklara yaslanır.
- **Seçim yanlılıkları aynı yönde birikir.** Erken durdurma, skorun ölçüldüğü
  fold'da ağaç sayısı seçer (ölçülen sapma %0.16); Optuna araması ~%0.3; SHAP
  geri eleme +0.0137 — üçü de iyimser yönde (`src/gridup` içindeki ölçümler).
  Nihai model kararı ayrılmış bir holdout veya LB doğrulaması ister.
- **İki sızıntıyı kendi denetimimiz yakaladı.** İlk prova aynı günün
  `effectedsubscribers` kolonunu feature almıştı; benchmark'ın ilk sürümünde ise
  ham kaydın `id` kolonu panel dolgusunun birebir kopyası çıktı (y==0 ile uyum
  0.9975) ve tüm skorları ~60 dk iyimser gösterdi. İkisi de çekişmeli denetimle
  bulundu, düzeltildi ve bu sayfadaki sayılar düzeltilmiş ölçümlerdir
  (prova 334.29, güncel harman 302.64). Sızıntı "bizde olmaz" denen şey değil,
  sistematik aranan şeydir.
- **Bu ölçümlerin kapsamı 2021–22 verisidir.** Ablasyon ve benchmark sayıları
  2021-05→2022-08 GDZ kaydında, `kesinti_dk` hedefi ve 47 ilçeyle ölçüldü.
  2026 yarışması muhtemelen farklı hedef ve 96 ilçeyle gelecek — "tatil zarar
  veriyor" gibi sonuçlar oraya taşınmaz, **1. günde yeni veride yeniden
  ölçülür** (`scripts/ablation_gercek.py` hazır, ~10 dk).
- **Public LB bir fold değildir.** LB rastgele bölmeyse zaman-temelli CV ile
  uyuşmayabilir; CV–LB korelasyonu düşükken LB'ye göre model seçmek shakeup'ta
  kaybettirir. Önce şema, sonra karar.

## 8 · Veri günü planı

Sıra ölçümden geliyor (`experiments/benchmark_gercek.json` → `gun1_recetesi`):

1. **Saat 0–2 · Keşif:** 01 notebook'u — panel, fold'lar, sızıntı duvarı.
   İlk iki saatin sonunda üç karar da verilmiş olmalı.
2. **İlk submission:** `catboost_mae` — 2023 birincisinin reçetesi, 3. dalga
   feature'larıyla gerçek GDZ'de kazanan tekil (MAE 304.89; hep-sıfır 366.97).
   Optimize etmeden önce LB'de bir sayı:
   ```python
   print(zero_baseline_score(y, metric="mae"))   # once bunu gectigini gor
   ```
   İki aşamalı + medyan kuralını aynı fold'larda ölçün (eşikli 317.04, medyan
   316.95): `q* = 1 − 0.5/p` kantil çözücüsü bedava ama kazancı feature setine
   bağlı — önceki dalgada eşikliyi 4.5 dk geçen fark, Hawkes feature'ları
   sinyali taşıyınca 0.1 dk'ya indi. Merdiven **koşullu** olmalı — marjinal
   `fit_quantile_ladder` burada ölçülmüş şekilde yanlış sonuç verir:
   ```python
   sonuc = fit_two_stage(train_features[FEATURES], y, folds, metric=METRIC)
   merdiven = fit_conditional_quantile_ladder(train_features[FEATURES], y, folds)
   tahmin = conditional_quantile_from_hurdle(sonuc.oof_probability, merdiven)
   ```
   Kalibrasyonu deneme — gerçek GDZ'de izotonik kalibrasyon Brier'i de MAE'yi de
   kötüleştirdi (`calibrate_positive_probability` ölçüp söyler).
3. **Feature'lar ablasyon sırasıyla:** önce lag (+22.34), sonra Hawkes bozunumu
   + toplu-olay payı (`add_event_decay_features` 3g/14g +
   `add_mass_event_features`, ikisi de ufuk şart; birlikte lgb_mae
   323.13→310.14), sonra hava (+2.47; `add_regional_aggregates` +
   `add_physical_derivatives`, ortalama değil `max`/quantile), komşu lag ucuzsa
   (`nearest_neighbours` + `add_neighbour_target_lag`, ufuk şart).
   `recency_activity_weights`'i Hawkes'la BİRLİKTE kullanmayın — aynı yenilik
   sinyali iki kanaldan verilince kaybettirdi (310.14→335.30, ölçüldü); önce
   feature'sız ölçün. Tatil/güneş en sona — gerçek veride negatif ölçüldüler.
   Her adımda kayma kontrolü:
   ```python
   sonuc = adversarial_validation(train_features[FEATURES], test_features[FEATURES])
   print(sonuc["auc"], sonuc["verdict"])   # AUC > 0.8 ise ayristiran feature'i cikar
   ```
4. **Aynı fold'larda** `lgb_mae`, `lgb_tweedie` ve `lgb_sqrt` eklenir
   (`sweep_count_objectives` ile) → TÜM üyeler üzerinde kapsam maskeli,
   kararlılık-cezalı hill-climb harmanı (`stability_penalty=0.5`; gerçek
   GDZ'de 302.64 — catboost_mae 0.75 + lgb_tweedie 0.25; "en iyi 3" kısayolu
   önceki dalgada 311.83 verdi — çeşitlilik kaliteden değerli).
5. **Hiperparametre araması** ancak harman oturduktan sonra — objective de arama
   uzayına girer: `tune_with_optuna(..., search_objective=True)`.
6. **Son gün:** çok tohumlu tam veri refit + jüri çıktıları:
   ```python
   tur = estimate_full_data_rounds(
       extract_best_iterations(result.models), n_folds=len(folds),
       mean_train_fraction=fold_train_fraction(folds, len(train)))
   final = multi_seed_refit(train_features[FEATURES], y, test_features[FEATURES],
                            params=params, n_estimators=tur, seeds=range(15))
   ```
   Feature elemesi gerekirse: `null_importance_filter` (dakikalar) →
   `shap_backward_selection` (saatler).

Bu sayılar gerçek GDZ provasında ölçüldü; yarışma verisinde **yeniden ölçülür**.
Plan sabit, sayılar değişebilir — değişirse karar da değişir.

## Jüri çıktıları

Bunları **son gün üretmeye kalkmayın** — pipeline'ın parçası olmalı.
Değerlendirmenin üçte ikisi notebook + sunum.

In [ ]:
from gridup.reporting import (
    business_impact, cv_fold_table, error_by_segment,
    feature_importance_table, model_footprint,
    plot_error_by_segment, plot_fold_scores, plot_prediction_timeline,
)

# Juri ciktilari OOF uzerinden hesaplanir: y_true = kapsanan satirlarin gercek
# degeri, y_pred = ayni satirlarin fold-disi tahmini. Kapsanmayan satirlar
# (purged semada ilk donem) DOLGU tasir, skora girmez.
kapsanan, y_pred = result.covered_predictions()
y_true = y[kapsanan]
grup_dilimi = train_features.loc[kapsanan, GROUP_COLUMN]

# 1 · Fold tablosu -- kararliligi gosterir
display(cv_fold_table(result))
plot_fold_scores(result); plt.show()

# 2 · Model NEREDE yaniliyor -- sunumun en ikna edici bolumu
segment_hatasi = error_by_segment(y_true, y_pred, grup_dilimi, metric=METRIC)
plot_error_by_segment(segment_hatasi, metric=METRIC); plt.show()

# 3 · Sinyal nereden geliyor -- 400 satirlik onem listesi yerine aile dagilimi
display(feature_importance_table(result, group_prefixes=(
    "tarih_", "tatil_", "komsu_", "bolge_", "sebep_")))

# 4 · Operasyonel maliyet -- juri bunu soruyor, modeli gercekten calistiracak
print(model_footprint(result.models, elapsed_seconds=result.elapsed_seconds))

# 5 · Is dili -- "MAE 2.95" degil, "ortalama 3 kesinti hatayla tahmin ediyoruz"
print(business_impact(y_true, y_pred, unit_label="kesinti")["ozet"])

## Sunum için not

Jüri koltuğunda **mühendisler ve iş birimleri** var, akademisyen değil.
2024 birincisinin sunumunun son üç slaydı tamamen iş değeriydi: açıklanabilir
çözüm, daraltılmış feature seti, ~25 MB model, yeni veriyle eğitilebilirlik.

Skor ilk 10'a sokar; bu bölüm ödülü belirler.